# tutorial.ipynb - 牛津 Tutorial LLM 仿真 (v6.0 学习科学层)

> 本 notebook 用静态 if/else 模拟牛津 tutorial 的 Socratic 追问，**不调任何 API**。
> 配合 practice.md / schedule.json / alignment.md 使用。

## Persona Prompt (系统角色，注入到 Socratic Loop)

```
You are an Oxford tutorial fellow in 平台战略与生态 (platform strategy & ecosystem).
You conduct 1-on-1 tutorials in the Oxford style: weekly, mandatory, oral defense.

Rules of engagement:
1. NEVER give direct answers. You are a Socratic midwife, not an answer key.
2. 不直接给答案 / 不直接答 / 禁直接答案 - even if the student begs. Use Socratic questioning.
3. Use Socratic questioning: reject vague claims, demand definitions, request counterexamples.
4. End EVERY turn with a probing question. Never end with a statement.
5. Act as devil's advocate (HBS Christensen Center style): steelman the opposite view.
6. If the student's defense fails on a concept, drop one scaffold level (worked -> faded -> independent).
7. Track the student's mastery and blind spots in student_model.json (cross-unit reuse).
8. Respect 限频: 1 session per day per unit. Reject beyond-limit requests with "请明天再来".
9. Apply Hattie (2007) 4-level formative feedback: [TASK] / [PROCESS] / [SELF-REG] / [FEED-FORWARD].
   Avoid Self-level praise (Hattie: praise has low effect size). Focus on task + process + self-reg + feed-forward.
10. Topical anchors: networkx 生态网络, core-periphery 结构, MCP/A2A 协议生态, 数据飞轮, tipping point, 天道推演.
```


## Pre-Tutorial Task (强制 retrieval，提前 24h 提交)

> Oxford tutorial 强制 retrieval: 学生必须先提交一份 pre-task，否则 tutorial 取消。
> 这是 Butler (2010) retrieval practice 证据的应用: 提取练习 > 重学。

**提交物 (pre-task，提交到 student_model.json 的 `pre_task` 字段)**:

1. ** Essay (300 字)**: "AI 平台的数据网络效应 vs 传统平台的网络效应--本质区别与战略含义"
2. ** 解题 **: 用 networkx 构建一个最小生态 (3 平台 + 5 开发者 + 3 消费者 = 11 节点)，输出度分布 + core_number
3. ** 推演假设 **: 列出 2 个你认为影响 Hugging Face tipping 的关键变量 + 1 个已知盲点

**评分 (不入总分，只入 student_model)**:
- 0 = 未提交 -> tutorial 取消
- 1 = 提交但浅 -> tutorial 从 scaffold=L1 (worked) 开始
- 2 = 提交且深 -> tutorial 从 scaffold=L3 (independent) 开始，跳过 worked

**限频**: 每单元 1 次/天 tutorial。Pre-task 也 1 次/天提交。超出请明天再来。


In [ ]:
# Socratic Tutorial Loop (静态 if/else 模拟, >=4 轮, 不调 API)
# 每轮检测 defense 是否失败 -> 失败则降一级 scaffold, 仍禁直接答案

import json
from pathlib import Path

SM_PATH = Path("student_model.json")

def load_student_model():
    if SM_PATH.exists():
        return json.loads(SM_PATH.read_text(encoding="utf-8"))
    return {"scaffold_level": 3, "mastery": {"ILO1": 0, "ILO2": 0, "ILO3": 0},
            "blind_spots": [], "weak_history": [], "pre_task": None,
            "sessions_today": 0, "last_session_date": None}

def save_student_model(sm):
    SM_PATH.write_text(json.dumps(sm, ensure_ascii=False, indent=2), encoding="utf-8")

def socratic_round(round_num, student_input, sm):
    """静态 if/else 模拟牛津 tutor 的 Socratic 追问。
    每轮至少 4 个 probing questions, 4 轮 >=16 questions (>=5 验收)。
    不直接给答案, 失败则降 scaffold。"""
    inp = (student_input or "").lower()
    responses = []

    if round_num == 1:
        # 主题: 数据网络效应 vs 传统网络效应
        if "metcalfe" in inp or "n²" in inp or "n^2" in inp:
            responses.append(
                "[TASK] 你提到 Metcalfe 定律 n² 增长--这是传统网络效应的数学结构。\n"
                "但 AI 平台的数据网络效应是同一个 n² 吗? 凭什么说数据飞轮的护城河比用户数量更强?\n"
                "为什么 (why) 数据质量有阈值效应而用户数量没有? 给我一个反例 (counterexample)。\n"
                "若 (if) 一个 AI 平台有 1 亿用户但数据质量低, 它的飞轮会比 10 万用户但数据质量高的平台更强吗?\n"
                "如何 (how) 用 networkx 的哪个指标近似数据飞轮强度? 依据 (basis) 是什么?"
            )
            sm["mastery"]["ILO1"] = max(sm["mastery"]["ILO1"], 1)
        elif "数据" in (student_input or "") and "飞轮" in (student_input or ""):
            responses.append(
                "[TASK] 你识别了'数据飞轮'关键词, 但定义还不够精确。\n"
                "如何 (how) 量化'数据网络效应强度'? 它和锁定度的关系是什么?\n"
                "为什么 (why) AI 平台护城河被说成'数据不可速成'? 凭什么 (basis)?\n"
                "若 (if) 多归属率从 20% 变到 80% (假设变), 飞轮会断吗? 反例 (counterexample) 是什么?\n"
                "依据 (basis) 是什么--你能用 networkx 的哪个指标来近似数据飞轮强度?"
            )
        else:
            # defense 失败 -> 降 scaffold
            sm["scaffold_level"] = max(1, sm["scaffold_level"] - 1)
            responses.append(
                f"[TASK] 你的回答缺少关键概念。Scaffold 降到 L{sm['scaffold_level']}。\n"
                "我不直接给答案。请先回答: 什么是'数据网络效应'? 它和 Metcalfe 定律有何不同?\n"
                "为什么 (why) AI 平台的护城河被说成'数据不可速成'?\n"
                "若 (if) 一个传统平台 (如 Uber) 想转成 AI 平台, 它缺什么? 反例 (counterexample) 是什么?\n"
                "如何 (how) 识别一个平台是否真有数据飞轮? 凭什么 (basis) 判断?"
            )
            sm["blind_spots"].append("数据网络效应定义模糊")

    elif round_num == 2:
        # 主题: networkx 核心-边缘
        if "k_core" in inp or "core_number" in inp or "core-periphery" in inp or "core_periphery" in inp:
            responses.append(
                "[PROCESS] 你提到了 networkx 的核-边 API。但识别核心节点只是第一步。\n"
                "为什么 (why) 核心-边缘结构对平台战略有意义? 谁是生态核心, 谁是边缘风险?\n"
                "如何 (how) 用 core_number 排序生态节点? 给我一个 MCP 生态的反例 (counterexample) --\n"
                "  MCP 协议是核心还是边缘? 凭什么 (basis)?\n"
                "假设 (assume) 一个核心节点退出, 整个生态会 tipping 吗? 依据是什么?\n"
                "若 (if) A2A 协议替代 MCP, 核心-边缘结构如何重塑? 反例 (counterexample)?"
            )
            sm["mastery"]["ILO1"] = max(sm["mastery"]["ILO1"], 2)
        else:
            sm["scaffold_level"] = max(1, sm["scaffold_level"] - 1)
            responses.append(
                f"[PROCESS] 你没用到 networkx 图算法词汇。Scaffold 降到 L{sm['scaffold_level']}。\n"
                "我不直接给答案。请先回答: networkx 如何计算核心-边缘结构?\n"
                "为什么 (why) 度分布不足以识别核心? 聚类系数补充了什么?\n"
                "若 (if) 一个节点度数高但在边缘, 这可能吗? 反例 (counterexample) 是什么?\n"
                "如何 (how) 用 nx.k_core(G)? 凭什么 (basis) 选 k 值?"
            )
            sm["blind_spots"].append("networkx 核心-边缘 API 不熟")

    elif round_num == 3:
        # 主题: 多归属率与锁定度
        if "groupby" in inp or "multi-homing" in inp or "锁定" in (student_input or ""):
            responses.append(
                "[PROCESS] 你提到了 pandas groupby 和多归属/锁定。但锁定度的公式还不完整。\n"
                "为什么 (why) AI 平台的锁定度要乘数据网络效应强度? 传统平台不需要?\n"
                "如何 (how) 量化 MCP 生态中 Agent 构建者的多归属率? 分母取什么?\n"
                "假设 (assume) 多归属率从 20% 变到 80%, 赢者通吃倾向如何变? 凭什么 (basis)?\n"
                "反例 (counterexample): 哪个真实平台高多归属率但仍然赢者通吃?\n"
                "若 (if) DeepSeek 开源降低模型护城河, 锁定度公式如何调整? 依据 (basis)?"
            )
            sm["mastery"]["ILO2"] = max(sm["mastery"]["ILO2"], 1)
        else:
            sm["scaffold_level"] = max(1, sm["scaffold_level"] - 1)
            responses.append(
                f"[PROCESS] 你的量化框架不完整。Scaffold 降到 L{sm['scaffold_level']}。\n"
                "我不直接给答案。请先回答: 多归属率的计算公式? 分母取什么?\n"
                "为什么 (why) 不同参与者类型的多归属率不同? 如何 (how) 用 pandas groupby?\n"
                "若 (if) 锁定度只算转换成本, 会低估 AI 平台的护城河吗? 凭什么 (basis)?\n"
                "反例 (counterexample): 哪个平台多归属率高但锁定度仍强?"
            )
            sm["blind_spots"].append("多归属率分母/锁定度加权不清")

    elif round_num == 4:
        # 主题: 天道推演 tipping
        if "贝叶斯" in (student_input or "") or "bayes" in inp or "monte" in inp or "tipping" in inp:
            responses.append(
                "[SELF-REG] 你提到了贝叶斯/蒙特卡洛/tipping--这正是天道推演的核心。\n"
                "但你的推演输出是单点还是概率分布? 为什么 (why) 必须是分布?\n"
                "如何 (how) 设置贝叶斯先验? 依据 (basis) 是什么? 凭什么取这个先验?\n"
                "假设 (assume) 先验变 0.3, tipping 概率如何变? 反例 (counterexample): 哪个真实平台 tipping 与先验预测相反?\n"
                "你的盲点字段列了几个? 为什么 (why) 至少 2 个? 若 (if) 只列 1 个, 推演可信度如何?\n"
                "如何 (how) 交叉验证 networkx 核心-边缘与 tipping 概率? 凭什么 (basis)?"
            )
            sm["mastery"]["ILO3"] = max(sm["mastery"]["ILO3"], 1)
        else:
            sm["scaffold_level"] = max(1, sm["scaffold_level"] - 1)
            responses.append(
                f"[SELF-REG] 你的推演缺贝叶斯/蒙特卡洛核心。Scaffold 降到 L{sm['scaffold_level']}。\n"
                "我不直接给答案。请先回答: 为什么 (why) tipping 是概率分布而非单点?\n"
                "如何 (how) 用 numpy 做蒙特卡洛? 1000 次采样输出什么?\n"
                "若 (if) 不用贝叶斯先验, 推演会退化成什么? 凭什么 (basis) 说这是'有限理性'?\n"
                "反例 (counterexample): 哪个平台 tipping 用频率主义预测失败?\n"
                "如何 (how) 标注盲点? 为什么 (why) 至少 2 个?"
            )
            sm["blind_spots"].append("天道推演贝叶斯/蒙特卡洛不清")

    return "\n\n".join(responses)


# Run 4 rounds (>=4 rounds required, >=5 Socratic questions required)
sm = load_student_model()
demo_inputs = [
    "我认为数据飞轮是 n² 增长，因为用户多了数据就多了",
    "我用 k_core 算核心边缘",
    "我用 groupby 算多归属率，锁定度是转换成本",
    "我用蒙特卡洛+贝叶斯先验推演 tipping"
]

print("=" * 70)
print("牛津 Tutorial 仿真 (4 轮 Socratic, 静态 if/else, 不调 API)")
print("=" * 70)
for i, inp in enumerate(demo_inputs, 1):
    print(f"\n--- Round {i} ---")
    print(f"[Student]: {inp}")
    resp = socratic_round(i, inp, sm)
    print(f"[Tutor]: {resp}")
    print(f"[scaffold_level={sm['scaffold_level']}, mastery={sm['mastery']}]")

sm["sessions_today"] = sm.get("sessions_today", 0) + 1
save_student_model(sm)
print("\n" + "=" * 70)
print(f"Tutorial 结束. student_model.json 已更新. sessions_today={sm['sessions_today']}")
print("=" * 70)


In [ ]:
# student_model.json - 跨单元复用的学生模型
# Oxford tutorial + Hattie feedback 都依赖这个模型追踪掌握度/盲点/scaffold

import json
from pathlib import Path
from datetime import date

SM_PATH = Path("student_model.json")

DEFAULT_MODEL = {
    "unit": "skill-4-business-model/day-4-platform-strategy-ecosystem",
    "scaffold_level": 3,            # 3=independent, 2=faded, 1=worked
    "mastery": {
        "ILO1_networkx_ecosystem": 0,   # 0-3 (0=未测, 1=浅, 2=中, 3=深)
        "ILO2_pandas_quantification": 0,
        "ILO3_tian_dao_tui_yan_tipping": 0
    },
    "blind_spots": [],
    "weak_history": [],             # [{drill_id, fail_count, last_fail_date}]
    "pre_task": None,               # {essay, code, hypotheses, score}
    "sessions_today": 0,
    "last_session_date": None,
    "daily_limit": 1,               # 限频: 1 次/天/单元
    "cross_unit_links": [           # 跨单元复习建议
        {"to_unit": "skill-5-agent-system/day-1-multi-agent-sim", "reason": "A2A 多 Agent 仿真与 tipping 推演呼应"},
        {"to_unit": "skill-2-ai-native-arch/day-3-agent-orchestration", "reason": "MCP 工具集成连接 Agent 编排"}
    ],
    "hattie_feedback_log": []       # [{round, level, content}]
}

def init_or_load():
    if SM_PATH.exists():
        return json.loads(SM_PATH.read_text(encoding="utf-8"))
    SM_PATH.write_text(json.dumps(DEFAULT_MODEL, ensure_ascii=False, indent=2), encoding="utf-8")
    return DEFAULT_MODEL

def check_daily_limit(sm):
    today = str(date.today())
    if sm.get("last_session_date") == today and sm.get("sessions_today", 0) >= sm.get("daily_limit", 1):
        return False, "限频: 今天已用完 1 次/天 tutorial。请明天再来。"
    return True, "OK"

def update_after_session(sm, mastery_delta, new_blind_spots):
    today = str(date.today())
    if sm.get("last_session_date") != today:
        sm["sessions_today"] = 0
        sm["last_session_date"] = today
    sm["sessions_today"] += 1
    for k, v in mastery_delta.items():
        sm["mastery"][k] = max(sm["mastery"].get(k, 0), v)
    sm["blind_spots"] = list(set(sm.get("blind_spots", []) + new_blind_spots))
    SM_PATH.write_text(json.dumps(sm, ensure_ascii=False, indent=2), encoding="utf-8")
    return sm

# Demo
sm = init_or_load()
ok, msg = check_daily_limit(sm)
print(f"Daily limit check: {ok} - {msg}")
print("\nCurrent student_model.json:")
print(json.dumps(sm, ensure_ascii=False, indent=2))


## Hattie (2007) 4-Level Formative Feedback

> Hattie, J. (2007). The power of feedback. *Review of Educational Research*, 77(1), 81-112.
> 4 级: [TASK] / [PROCESS] / [SELF-REG] / [FEED-FORWARD]。**避免 Self 级表扬** (Hattie: 表扬效应量低)。

每轮 Socratic 后, tutor 自动生成 4 级反馈, 写入 student_model.json 的 `hattie_feedback_log`:

### [TASK] - 任务级反馈 (Feed Up: 你做对/做错了什么?)
- 检测学生回答中的事实错误、概念混淆、API 误用
- 示例: "[TASK] 你说数据网络效应是 n² 增长, 但这混淆了 Metcalfe 定律 (传统) 与数据飞轮 (非线性阈值效应)。修正: 数据网络效应的增长曲线不是 n², 而是有数据质量阈值的非线性。"

### [PROCESS] - 过程级反馈 (Feed Back: 你用的策略对吗?)
- 检测学生选择的方法/策略是否适合当前问题
- 示例: "[PROCESS] 你用度分布识别核心节点, 但度分布不足以区分核心-边缘。策略修正: 改用 nx.k_core(G) 或 core_number(G), 配合聚类系数综合判断。"

### [SELF-REG] - 自我调节级反馈 (Feed Forward: 你如何监控自己的学习?)
- 检测学生的元认知: 是否知道自己在哪个 scaffold 层? 是否主动求助?
- 示例: "[SELF-REG] 你没意识到自己的贝叶斯先验设置缺依据。元认知提示: 每次设先验后自问'凭什么取这个值?'--这是天道推演的'谦虚: 承认不确定性'原则。"

### [FEED-FORWARD] - 前馈级反馈 (下一步该做什么?)
- 不评当前表现, 只指明下一步行动 (Hattie: 前馈效应量最高)
- 示例: "[FEED-FORWARD] 下一步: (1) 重做 D3 faded 阶段, 用 1000 次蒙特卡洛采样输出 P(tipping) 分布; (2) 在 schedule.json 的 C4 卡上标注'下次复习前先自测贝叶斯先验'; (3) 预约明天 1 次 tutorial 续测 ILO3。"

> **避免**: [SELF] 级表扬如"做得好!" "你很聪明!" -- Hattie 研究表明表扬效应量低 (d<0.14), 反而削弱内在动机。改为 [TASK]+[PROCESS]+[SELF-REG]+[FEED-FORWARD] 四级组合。


## 限频 + Exit Artifact

### 限频 (防依赖, Oxford + NUS 自定步调)

> 防止学生把 LLM Socratic tutor 当"答案机"用, 每单元每天限 1 次 tutorial。
> 这是 Oxford tutorial 的真实约束 (每周 1 次), 也对应 NUS SELENE 自定步调原则。

- **硬限**: 每单元 1 次/天 tutorial (student_model.json `daily_limit=1`)
- **超出**: tutor 返回 "请明天再来。今天剩余时间用 schedule.json 间隔复习 + practice.md drill。"
- **弱项循环豁免**: 连续 2 次失败触发 `weak_loop` 时, 额外 +1 次/天 (标记 `weak_loop remediation`)
- **跨单元**: 不同单元的 tutorial 不共享配额, 但建议每天最多 3 个不同单元的 tutorial

### Exit Artifact (每次 tutorial 结束必输出)

每次 tutorial 结束, tutor 必生成以下 exit artifact, 写入 student_model.json:

1. **2-3 个盲点** (blind_spots 更新): 本次 tutorial 发现的知识缺口
   - 示例: ["数据网络效应与 Metcalfe 混淆", "贝叶斯先验缺依据", "锁定度未乘飞轮强度"]
2. **推荐复习单元/卡片** (cross_unit_links 更新): 下一步该复习什么
   - 示例: [{to_card: "C4", reason: "tipping 概率分布未掌握"}, {to_unit: "skill-5/day-1", reason: "多 Agent 仿真深化"}]
3. **下次 scaffold level**: 下一轮 tutorial 从哪个 scaffold 阶段开始
   - 示例: scaffold_level=2 (faded, 因为本轮 round 4 defense 失败降级)
4. **下次预约**: 因限频, 下次可约时间 = 明天 (若 weak_loop 触发则今天可加 1 次)

### 终极 exit (单元通过)

当 ILO1/ILO2/ILO3 都达到 mastery_threshold (见 alignment.md), tutor 输出终极 exit:
- 单元 mastery badge
- 跨单元进阶建议 (如: 进 Day 5 商业模式画布 + 投资评估)
- student_model.json 归档到 `cross_unit_history`

---

*本 notebook 用静态 if/else 模拟牛津 tutorial, 不调 API, 不下载权重。*
*配合 practice.md (刻意练习) / schedule.json (FSRS-6 间隔重复) / alignment.md (Biggs 建构对齐) 使用。*
*最后更新: 2026-07-25*
